# 1a. Filter for high-quality genomes to download

In this notebook, we will use __`pyphylon`__'s `download` and `qcqa` modules to select candidate genomes to download for pangenome generation.

In this example we will select genomes for download from [BV-BRC](https://www.bv-brc.org/)

In [ ]:
import os
import subprocess
import time
import zipfile
import numpy as np
import pandas as pd

import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker

from tqdm.notebook import tqdm

plt.rcParams["figure.dpi"] = 150
sns.set_context("paper")
sns.set_style("whitegrid")

In [ ]:
## NCBI metadata -- filtered for PA strains, only complete genomes

ncbi_metadata = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/metadata/ncbi_genome_metadata_1_15_2026.tsv', sep = '\t')
ncbi_metadata = ncbi_metadata.loc[ncbi_metadata['Assembly Level'] == 'Complete Genome']

In [ ]:
# Coerce QC metrics to numeric for plotting/filtering
#L50 values downloaded using NCBI cli (paeruginosa_L50.tsv) -- all L50 values = 1
qc_columns = {
    "Assembly Stats Contig N50": "contig_n50",
    "Assembly Stats Total Sequence Length": "total_sequence_length",
    "Assembly Stats GC Percent": "gc_percent",
    "CheckM completeness": "checkm_completeness",
    "CheckM contamination": "checkm_contamination",
    "Annotation Count Gene Protein-coding": "cds",
    "Assembly Stats Number of Contigs": "contigs"
}

for raw_col, clean_col in qc_columns.items():
    if raw_col not in ncbi_metadata.columns:
        raise ValueError(f"Missing expected column: {raw_col}")
    ncbi_metadata[clean_col] = pd.to_numeric(ncbi_metadata[raw_col], errors="coerce")

if "ANI Check status" not in ncbi_metadata.columns:
    raise ValueError("Missing expected column: ANI Check status")

ncbi_metadata.head()

In [ ]:
# Initial unfiltered strain plot
h = sns.jointplot(
    data=ncbi_metadata,
    x="total_sequence_length",
    y="cds",
    # hue="genome_status",
    alpha=0.75,
    height=4
)

# h.ax_joint.legend(
#     title='BV-BRC\nstrain type',
# )

h.ax_joint.set_xlabel("genome length")
h.ax_joint.set_ylabel("NCBI predicted gene count")
plt.show()

In [ ]:
######### Find the scaffold N50 score of the reference genome for the organism of interest
######### Either visit the NCBI website or retrieve it using the following method (~20 seconds)
######### (https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000006765.1/)
scaffold_n50 = 6.3e6

In [ ]:
fig, ax = plt.subplots()

# Set threshold as 0.85 * Scaffold N50 score
species_ref_n50 = scaffold_n50
min_thresh_n50 = int(0.85 * species_ref_n50)

# Most (if not all) Complete sequences pass this threshold
sns.histplot(ncbi_metadata.contig_n50.dropna().astype('int'), ax=ax)
plt.axvline(x=min_thresh_n50, color='#ff00ff', linestyle='--')

In [ ]:
ncbi_metadata = ncbi_metadata.dropna(subset=['contig_n50', 'total_sequence_length', 'gc_percent',
       'checkm_completeness', 'checkm_contamination', 'cds'])

In [ ]:
def plot_metric_distribution(df, column, title, bins=50, log_x=False, vlines=None, scale_millions=False):
    # Plot a QC metric distribution with optional threshold lines.
    fig, ax = plt.subplots()
    data = df[column].dropna()

    if log_x:
        data = data[data > 0]
        ax.set_xscale("log")

    sns.histplot(data, bins=bins, ax=ax)

    if scale_millions:
        if log_x:
            ax.xaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1, 2, 3, 4, 5, 6, 7, 8, 9)))
        else:
            ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=8))

        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: x / 1e6))
        ax.set_xlabel(f"{column} (1e6)")
    ax.set_title(title)
    ax.set_xlabel(column)

    if vlines:
        for label, value in vlines.items():
            ax.axvline(value, linestyle="--", linewidth=1)
            ax.text(value, ax.get_ylim()[1] * 0.9, label, rotation=90, va="top")

    plt.show()

In [ ]:
# Plot QC metric distributions
plot_metric_distribution(
    ncbi_metadata,
    "contig_n50",
    "Contig N50 Distribution",
    bins=1000,
    log_x=False,
    scale_millions=True,
)


plot_metric_distribution(
    ncbi_metadata,
    "total_sequence_length",
    "Total Sequence Length Distribution",
    bins=1000,
    log_x=False,
    scale_millions=True,
)

plot_metric_distribution(
    ncbi_metadata,
    "gc_percent",
    "GC Percent Distribution",
    bins=50,
)

plot_metric_distribution(
    ncbi_metadata,
    "checkm_completeness",
    "CheckM Completeness Distribution",
    bins=1000,
)

plot_metric_distribution(
    ncbi_metadata,
    "checkm_contamination",
    "CheckM Contamination Distribution",
    bins=1000,
)

plot_metric_distribution(
    ncbi_metadata,
    "cds",
    "CDS Count Distribution",
    bins=1000,
)

In [ ]:
# Set thresholds based on the plots
thresholds = {
    "ani_check_status": "OK",
    "contig_n50_min": scaffold_n50 * 0.85, #### scaffold N50 for ref strain on NCBI * 0.85
    "total_sequence_length_min": 5e6,
    "total_sequence_length_max": None,
    "gc_percent_min": 65, ### typically ~65% for PA species
    "gc_percent_max": None,
    "checkm_completeness_min": 95,
    "checkm_contamination_max": 3,
    "cds_min" : 5e3,
}

In [ ]:
def filter_with_report(df, thresholds):
    steps = []
    current = df.copy()

    def apply_step(name, mask):
        nonlocal current
        start = len(current)
        current = current[mask]
        end = len(current)
        steps.append({
            "step": name,
            "start": start,
            "filtered_out": start - end,
            "remaining": end,
        })

    if thresholds.get("ani_check_status") is not None:
        apply_step(
            "ANI Check status",
            current["ANI Check status"] == thresholds["ani_check_status"],
        )

    if thresholds.get("contig_n50_min") is not None:
        apply_step(
            "Contig N50 min",
            current["contig_n50"] >= thresholds["contig_n50_min"],
        )

    if thresholds.get("total_sequence_length_min") is not None:
        apply_step(
            "Total sequence length min",
            current["total_sequence_length"] >= thresholds["total_sequence_length_min"],
        )

    if thresholds.get("total_sequence_length_max") is not None:
        apply_step(
            "Total sequence length max",
            current["total_sequence_length"] <= thresholds["total_sequence_length_max"],
        )

    if thresholds.get("gc_percent_min") is not None:
        apply_step(
            "GC percent min",
            current["gc_percent"] >= thresholds["gc_percent_min"],
        )

    if thresholds.get("gc_percent_max") is not None:
        apply_step(
            "GC percent max",
            current["gc_percent"] <= thresholds["gc_percent_max"],
        )

    if thresholds.get("checkm_completeness_min") is not None:
        apply_step(
            "CheckM completeness min",
            current["checkm_completeness"] >= thresholds["checkm_completeness_min"],
        )

    if thresholds.get("checkm_contamination_max") is not None:
        apply_step(
            "CheckM contamination max",
            current["checkm_contamination"] <= thresholds["checkm_contamination_max"],
        )

    if thresholds.get("cds_min") is not None:
        apply_step(
            "CDS count min",
            current["cds"] >= thresholds["cds_min"],
    
        )
    
    # if thresholds.get("contig_count") is not None:
    #     apply_step(
    #         "Contig Count",
    #         current["contigs"] == thresholds["contig_count"],
    #     )

    report = pd.DataFrame(steps)
    return current, report

filtered_ncbi_metadata, filtration_report = filter_with_report(ncbi_metadata, thresholds)

print(f"Prefilter: {len(ncbi_metadata)} genomes")
print(f"Postfilter: {len(filtered_ncbi_metadata)} genomes")

filtration_report

In [ ]:
# Save filtered metadata to a TSV in the current folder
filtered_tsv_path = "/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1a_ncbi_genome_metadata_filtered.tsv"
filtered_ncbi_metadata.to_csv(filtered_tsv_path, sep="\t", index=False)
filtered_tsv_path

## Download filtered genomes via script

Run the script from this folder using the filtered TSV created above:

```bash
python ./Download_Genomes.py /mnt/craig/pan_phylon/P_aeruginosa/data/interim/1a_ncbi_genome_metadata_filtered.tsv --output-dir /mnt/craig/pan_phylon/P_aeruginosa/data/raw/genomes/ncbi
```